# DINO dense degradation sweep: all Drive checkpoints

This Colab notebook evaluates every recognizable DINO ViT-S/16 checkpoint in a Google Drive checkpoint folder. It follows the dense degradation protocol used in *Exploring Structural Degradation in Dense Representations for Self-supervised Learning*: frozen backbone, projector removed, last-layer patch embeddings, lightweight linear semantic segmentation head, PASCAL VOC mIoU curve.

It also runs the patch-level diagnostic suite requested during supervision: DSE class separability/effective rank, patch feature magnitude histograms, CLS-to-patch cosine, CLS attention maps, fixed-query patch similarity maps, and deterministic fixed-basis PCA patch-feature maps.

By default this notebook scans `/content/drive/MyDrive/dinocheckpoint` and runs all `checkpoint*.pth` files it can identify. Update the paths in the configuration cell before running if your Drive layout differs.


## 1. Mount Google Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf /content/dino
!git clone https://github.com/xbz123/dino-dense-degradation.git /content/dino
%cd /content/dino
!git rev-parse HEAD

## 2. Configure paths and evaluation settings

In [ ]:
from pathlib import Path

# Google Drive folder containing checkpoint*.pth files.
# The notebook auto-discovers every recognizable checkpoint in this folder, including names like:
# checkpoint03.pth, checkpoint0020.pth, checkpoint0125.pth, checkpoint0210.pth, checkpoint215.pth.
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/dinocheckpoint'

# Optional filter. Keep None to evaluate every checkpoint found in DRIVE_CHECKPOINT_DIR.
# Example for a quick smoke test: CHECKPOINT_EPOCH_FILTER = [180, 190, 200, 210, 215]
CHECKPOINT_EPOCH_FILTER = None

# ImageNet-style image folder used for DSE/patch diagnostics. This should match the pretraining data distribution.
# It must be readable by torchvision.datasets.ImageFolder: root/class_name/image.jpg.
DSE_IMAGE_ROOT_CANDIDATES = [
    '/content/drive/MyDrive/imagenet100/train',
    '/content/drive/MyDrive/ImageNet100/train',
]
DSE_IMAGE_ROOT = next((p for p in DSE_IMAGE_ROOT_CANDIDATES if Path(p).is_dir()), DSE_IMAGE_ROOT_CANDIDATES[0])

# Base output directory. Raw/L2 validation writes into OUTPUT_ROOT/to_epoch_XXXX_raw_l2.
OUTPUT_ROOT = '/content/drive/MyDrive/dino_dense_degradation_eval'
OUTPUT_RUN_SUFFIX = 'raw_l2'
RUN_VOC_EVAL = False  # Keep False to reuse existing VOC results and only rerun raw/L2 patch diagnostics.
WORK_CKPT_DIR = '/content/dino_eval_checkpoints'
RUN_OUTPUT_ROOT = None
BASE_RUN_OUTPUT_ROOT = None
VOC_JSON_FOR_REPORT = None

# Paper-aligned settings. If Colab runs out of memory, reduce VOC_LINEAR_BATCH_SIZE to 64 or 32.
VOC_IMG_SIZE = 336
VOC_LINEAR_BATCH_SIZE = 128
VOC_LINEAR_LR = 0.01 * VOC_LINEAR_BATCH_SIZE / 256
VOC_LINEAR_EPOCHS = 15

# Patch/DSE diagnostics. Paper-style DSE uses 2048 sampled pretraining images.
# For a smoke test, set NUM_DSE_IMAGES=128 and PATCH_DSE_GROUP_STRIDE=8.
NUM_DSE_IMAGES = 2048
NUM_VIS_IMAGES = 6
PATCH_DIAG_BATCH_SIZE = 32
PATCH_DIAG_NUM_WORKERS = 2
CHECKPOINT_KEY = 'teacher'
MAX_SPECTRUM_TOKENS = 30000
MAX_KMEANS_TOKENS = 12000
PATCH_DSE_GROUP_STRIDE = 1
STRICT_INTERNAL_EPOCH = False
STRICT_INTERNAL_ARG = '--strict_internal_epoch' if STRICT_INTERNAL_EPOCH else ''

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
Path(WORK_CKPT_DIR).mkdir(parents=True, exist_ok=True)
print('checkpoint dir:', DRIVE_CHECKPOINT_DIR)
print('checkpoint filter:', CHECKPOINT_EPOCH_FILTER)
print('DSE image root:', DSE_IMAGE_ROOT)
print('base output root:', OUTPUT_ROOT)
print('output suffix:', OUTPUT_RUN_SUFFIX)
print('run VOC eval:', RUN_VOC_EVAL)
print('VOC lr:', VOC_LINEAR_LR)


## 3. Auto-discover, copy, and verify checkpoints from Google Drive

The evaluator expects epoch numbers in filenames. This cell scans the Drive folder, infers the epoch from names like `checkpoint0120.pth` or `checkpoint215.pth`, optionally uses `checkpoint.pth` if it has an internal epoch and no epoch-named duplicate exists, and normalizes everything to `checkpoint####.pth` under `/content/dino_eval_checkpoints`.


In [ ]:
import os, json, shutil
from pathlib import Path
from dense_eval_utils import build_run_output_root, discover_checkpoint_files

assert os.path.isdir(DRIVE_CHECKPOINT_DIR), DRIVE_CHECKPOINT_DIR
print('=== Drive checkpoint files ===')
all_drive_files = sorted(os.listdir(DRIVE_CHECKPOINT_DIR))
print('\n'.join(all_drive_files[:500]))

selected = discover_checkpoint_files(DRIVE_CHECKPOINT_DIR, epoch_filter=CHECKPOINT_EPOCH_FILTER)
assert selected, f'No recognizable checkpoint*.pth files found in {DRIVE_CHECKPOINT_DIR}'

# Clear old normalized checkpoint files from this runtime so repeated runs cannot mix old selections.
for name in os.listdir(WORK_CKPT_DIR):
    if name.startswith('checkpoint') and name.endswith('.pth'):
        os.remove(os.path.join(WORK_CKPT_DIR, name))

prepared = []
print('=== Selected checkpoints ===')
for item in selected:
    dst = Path(WORK_CKPT_DIR) / f'checkpoint{item.epoch:04d}.pth'
    shutil.copy2(item.path, dst)
    print(f"{item.epoch:>4}: {item.path.name} -> {dst} | internal_epoch={item.internal_epoch} | {item.size_mb:.1f} MB")
    prepared.append({
        'epoch': item.epoch,
        'source': str(item.path),
        'normalized': str(dst),
        'internal_epoch': item.internal_epoch,
        'size_mb': item.size_mb,
    })

with open(os.path.join(WORK_CKPT_DIR, 'selected_checkpoints.json'), 'w') as f:
    json.dump(prepared, f, indent=2)

SELECTED_EPOCHS = [item['epoch'] for item in prepared]
FINAL_EPOCH = max(SELECTED_EPOCHS)
BASE_RUN_OUTPUT_ROOT = str(build_run_output_root(OUTPUT_ROOT, SELECTED_EPOCHS))
RUN_OUTPUT_ROOT = str(build_run_output_root(OUTPUT_ROOT, SELECTED_EPOCHS, suffix=OUTPUT_RUN_SUFFIX))
VOC_JSON_FOR_REPORT = (
    f'{RUN_OUTPUT_ROOT}/voc_all_checkpoints/voc_miou_results.json'
    if RUN_VOC_EVAL
    else f'{BASE_RUN_OUTPUT_ROOT}/voc_all_checkpoints/voc_miou_results.json'
)
os.makedirs(RUN_OUTPUT_ROOT, exist_ok=True)
shutil.copy2(
    os.path.join(WORK_CKPT_DIR, 'selected_checkpoints.json'),
    os.path.join(RUN_OUTPUT_ROOT, 'selected_checkpoints.json'),
)
print('=== Prepared epochs ===')
print(SELECTED_EPOCHS)
print('final epoch:', FINAL_EPOCH)
print('base run output root:', BASE_RUN_OUTPUT_ROOT)
print('run output root:', RUN_OUTPUT_ROOT)
print('VOC json for report:', VOC_JSON_FOR_REPORT)
print('=== Prepared files ===')
print('\n'.join(sorted(os.listdir(WORK_CKPT_DIR))))


## 4. Patch the VOC linear evaluator to use the paper-style Adam optimizer

In [ ]:
eval_path = '/content/dino/eval_voc_dense.py'
txt = Path(eval_path).read_text()
old = "optimizer = torch.optim.SGD(head.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)"
new = "optimizer = torch.optim.Adam(head.parameters(), lr=lr)"
if old in txt:
    txt = txt.replace(old, new)
    Path(eval_path).write_text(txt)
    print('Patched optimizer: SGD -> Adam')
else:
    print('Optimizer patch was already applied or source changed.')

## 5. Run PASCAL VOC frozen-backbone linear segmentation

This is the main dense degradation curve. It trains a linear head for every normalized checkpoint discovered from Drive, so runtime scales with the number of checkpoints.


In [ ]:
import subprocess

if RUN_VOC_EVAL:
  subprocess.run([
      'python', '/content/dino/eval_voc_dense.py',
      '--ckpt_dir', WORK_CKPT_DIR,
      '--voc_root', '/content/voc_data',
      '--arch', 'vit_small',
      '--patch_size', '16',
      '--img_size', str(VOC_IMG_SIZE),
      '--train_epochs', str(VOC_LINEAR_EPOCHS),
      '--lr', str(VOC_LINEAR_LR),
      '--batch_size', str(VOC_LINEAR_BATCH_SIZE),
      '--feature_dtype', 'float16',
      '--output_dir', f'{RUN_OUTPUT_ROOT}/voc_all_checkpoints',
  ], check=True)
else:
  print('Skipping VOC eval; reusing:', VOC_JSON_FOR_REPORT)
  assert Path(VOC_JSON_FOR_REPORT).is_file(), VOC_JSON_FOR_REPORT


## 6. Run DSE, patch statistics, attention, and fixed-query visualizations

This step uses the repository scripts directly instead of writing temporary code inside the notebook. The outputs include per-checkpoint metrics, histograms, qualitative fixed-image maps, fixed-image metadata, shared deterministic PCA-basis maps, query-point metadata, a combined summary figure, and a Markdown report.


In [ ]:
!python /content/dino/analyze_patch_statistics.py   --ckpt_dir {WORK_CKPT_DIR}   --image_root {DSE_IMAGE_ROOT}   --out {RUN_OUTPUT_ROOT}/patch_attention_dse_all_checkpoints   --arch vit_small   --patch_size 16   --checkpoint_key {CHECKPOINT_KEY}   --num_metric_images {NUM_DSE_IMAGES}   --num_vis_images {NUM_VIS_IMAGES}   --batch_size {PATCH_DIAG_BATCH_SIZE}   --num_workers {PATCH_DIAG_NUM_WORKERS}   --max_spectrum_tokens {MAX_SPECTRUM_TOKENS}   --max_kmeans_tokens {MAX_KMEANS_TOKENS}   --dse_group_stride {PATCH_DSE_GROUP_STRIDE}   --seed 0 {STRICT_INTERNAL_ARG}


## 7. Plot combined curves and write the run report


In [ ]:
!python /content/dino/plot_dense_diagnostics.py   --summary_csv {RUN_OUTPUT_ROOT}/patch_attention_dse_all_checkpoints/patch_attention_dse_summary.csv   --voc_json {VOC_JSON_FOR_REPORT}   --out_dir {RUN_OUTPUT_ROOT}/figures

!python /content/dino/make_summary_report.py   --summary_csv {RUN_OUTPUT_ROOT}/patch_attention_dse_all_checkpoints/patch_attention_dse_summary.csv   --voc_json {VOC_JSON_FOR_REPORT}   --out {RUN_OUTPUT_ROOT}/summary_report.md


## 8. Inspect outputs

All persistent raw/L2 validation outputs are saved under `RUN_OUTPUT_ROOT`, for example `/content/drive/MyDrive/dino_dense_degradation_eval/to_epoch_0215_raw_l2/`. VOC is skipped by default and read from the matching base run, for example `/content/drive/MyDrive/dino_dense_degradation_eval/to_epoch_0215/voc_all_checkpoints/voc_miou_results.json`.


In [ ]:
!echo '=== Selected checkpoints ==='
!cat {RUN_OUTPUT_ROOT}/selected_checkpoints.json || true

!echo '=== VOC results ==='
!echo {VOC_JSON_FOR_REPORT}
!cat {VOC_JSON_FOR_REPORT} || true

!echo '=== DSE / patch / attention summary ==='
!ls -lh {RUN_OUTPUT_ROOT}/patch_attention_dse_all_checkpoints || true
!cat {RUN_OUTPUT_ROOT}/patch_attention_dse_all_checkpoints/patch_attention_dse_summary.json || true

!echo '=== combined figures and report ==='
!ls -lh {RUN_OUTPUT_ROOT}/figures || true
!ls -lh {RUN_OUTPUT_ROOT}/figures/fig_raw_vs_l2_*.png || true
!cat {RUN_OUTPUT_ROOT}/summary_report.md || true

!echo '=== sample qualitative figures ==='
!find {RUN_OUTPUT_ROOT}/patch_attention_dse_all_checkpoints -maxdepth 2 -type f | head -80 || true
